In [1]:
import numpy as np

# Preliminary Functions

In [2]:
# Compute the power of a^n modulo m
def fast2Power(a,n,m):
    res = 1
    while n>0:
        if n%2 == 1: # If the bit is 1 multiply by the corresponding square
            res = (res*a) % m
        a = (a*a) % m
        n = n//2
    return res

In [3]:
# Output [r,s,t] satisfying s*a+t*b=r=gcd(a,b)
def extendedGCD(a,b):
    r0,r = a,b
    s0,s = 1,0
    t0,t = 0,1
    while (r>0):
        tempr,temps,tempt = r,s,t
        q = r0 // r
        r,s,t = r0-q*r,s0-q*s,t0-q*t
        r0,s0,t0 = tempr,temps,tempt
    return [r0,s0,t0]

In [4]:
# Output the multiplicative inverse of a modulo p
def multInverse(a,p):
    result = extendedGCD(a,p)
    if result [0]!=1: # Error message if a and p are not relatively prime
        s = " Numbers need to be relatively prime "
        return s
    inv = result [1]% p
    return inv

Implementing the algorithms 9 and 10 described in NIST.FIPS.204 Dilithium

In [5]:
#takes as input the integer x and the power alpha
#returns a list of length alpha corresponding to the bit representation of x mod 2^alpha

def integerToBits(x,alpha):
    result_list = []
    for i in range(alpha):
        result_list.append(x % 2)
        x = x // 2
    return result_list

In [6]:
# takes as input a list of bits and the alpha
# returns the corresponding integer

def bitsToInteger(bitList,alpha):
    x = 0
    for i in range(1,alpha+1):
        x = 2*x + bitList[alpha-i]
    return x

Implementing a generalized version of algorithm 43 from NIST.FIPS.204 Dilithium

In [7]:
# outputs the number (in decimal base) that one gets when reversing the log2(n)-bit binary expansion of m
def bitRev(n,m):
    alpha = n.bit_length() - 1  # 2^alpha = n; alpha = log2(n)
    bitList = integerToBits(m,alpha)
    bitReversed = [0]*(alpha)
    for i in range(alpha):
        bitReversed[i] = bitList[(alpha-1)-i]
    r = bitsToInteger(bitReversed,alpha)
    return r

In [8]:
# generates a random polynomial of degree at most n-1 with coefficients in Z_q
def genRandPoly(n,q):
    rng = np.random.default_rng()
    f = rng.integers(low=0,high=q,size=n)
    return f

In [9]:
# finds a generator w of Z_q* given the prime factors of q-1 as a list

def findGenerator(q,primeList):
    for w in range(2,q-1):  # possible generators
        flag = False
        for p in primeList:
            if fast2Power(w,(q-1)//p,q) == 1:
                flag = True
                break

        if flag == False:
            return w

# Schoolbook Multiplication in Z_q[x]/(x^n-a)

Following conceptual review on NTT but generalizing for any a instead of just +- 1.

In [10]:
# input are two polynomials f and g of degree at most n-1 with coefficients in Z_q
# the algorithm calculates the product c = f*g in Z_q[x]/(x^n-a)
# f and g are numpy arrays and the resulting output is also a numpy array

def schoolMult(f,g,n,q,a):
    c = np.zeros(n,dtype=int)
    for k in range(n):
        c_k = 0
        c_k_temp = 0  # the sum that is multiplied by a since the power of x is at least n-1
        for i in range(0,k+1):
            c_k = (c_k + f[i]*g[k-i]) % q
        for i in range(k+1,n):
            c_k_temp = (c_k_temp + ((f[i]*g[k+n-i])%q)) % q
        c_k_temp = (a*c_k_temp) % q    
        c[k] = (c_k + c_k_temp) % q
    return c

# NTT

## Precomputation of arrays

In [11]:
# computes the array with the powers of w and the array with the powers of w for the INTT
# notice the ordering of the array for the INTT
# w is a generator of Z_q*
# n is power of 2, q is a prime with n|(q-1) and k is the exponent such that w^(n*k) = a
# output is a tuple NTT_array,INTTarray

def compute_arrays(w,n,q,k):
    # the fraction we will always use in the exponents
    fraction = (q-1) // n 
    
    # arrays with the powers of w; length n because 0 at position 0 for NTT algorithm
    NTTarray = np.zeros(n,dtype=int) 
    INTTarray = np.zeros(n,dtype=int) 

    # the last vertical layer from top, it is log2(n) since we start with 1
    last_layer = n.bit_length() - 1 
    
    # indicates the position in the NTTarray and INTTarray which have to be filled next
    array_position = 1 
    INTTarray_position = 1 

    for layer_from_top in range(1,last_layer+1):
        #horizontal length of current layer
        layer_length = 2**(layer_from_top-1)
        # k is multiplied with the same integers in the whole vertical layer
        k_value = bitRev(n,2**(layer_from_top-1))*k 
        # setting the INTTarray_position to the rightmost position in the current vertical layer
        INTTarray_position = array_position + layer_length - 1
        
        for horizontal_position in range(2**(layer_from_top-1)):
            # the exponents of w for the NTT_array
            exp_NTT = (bitRev(n,2*horizontal_position)*fraction + k_value) % (q-1)
            # adding (q-1)//2 in the exponent of w leads to a multiplication by -1
            exp_INTT = ((q-1)//2 - exp_NTT) % (q-1)
            
            NTTarray[array_position] = fast2Power(w,exp_NTT,q)
            INTTarray[INTTarray_position] = fast2Power(w,exp_INTT,q)
            array_position += 1 
            # moving towards the left in the vertical layer
            INTTarray_position -= 1 

    return NTTarray,INTTarray

## Computing the NTT (generalization of Algorithm 41 in NIST.FIPS.204 Dilithium)

In [12]:
# n is as usual the dimension (a power of two), q the prime,
# f is the polynomial in Z_q[x]/(x^n-w^(n*k)) represented by a list or array of length n
# w_array is the array created by w_array(w,n,q,k)

# computes the NTT transform defined by the w_array
def NTT(f,n,q,NTT_array):
    m = 0
    length = n // 2
    while length >= 1:
        start = 0
        while start < n:
            m += 1
            z = NTT_array[m]
            for j in range(start,start+length):
                t = (z*f[j+length]) % q
                f[j+length] = (f[j]-t) % q
                f[j] = (f[j]+t) % q
            start = start + 2*length
        length //= 2 
    return f

## Computing the Inverse NTT (INTT) (generalization of Algorithm 42 in NIST.FIPS.204 Dilithium)

In [13]:
# v is a numpy array
# n is as usual the dimension (a power of two), q the prime
# INTTarray is the array with the powers of w that we need for the inverseNTT (created by compute_arrays(w,n,q,k))
# n_inv is the multiplicative inverse of n modulo q

def INTT(v,n,q,n_inv,INTTarray):
    m = n
    length = 1
    while length < n:
        start = 0
        while start < n:
            m -= 1
            z = -INTTarray[m]
            for j in range(start,start+length):
                t = v[j]
                v[j] = (t+v[j+length]) % q
                v[j+length] = (t-v[j+length]) % q
                v[j+length] = (z*v[j+length]) % q
            start = start + 2*length
        length *= 2
    v = (n_inv*v) % q
    return v    

## Addition and multiplication for NTT form

In [14]:
# computes the sum of v and u
# v and u are given in NTT form (as numpy arrays)
# the output is their sum in NTT form (as a numpy array)
# q is the prime as usual

def addNTT(v,u,q):
    return (v+u) % q

In [15]:
# computes the component-wise product of v and u
# v and u are given in NTT form (as numpy arrays)
# the output is their product in NTT form (as a numpy array)
# q is the prime as usual

def multiplyNTT(v,u,q):
    return (v*u) % q

## Polynomial Multiplication using NTT

In [16]:
# polynomial multiplication using NTT
def polyMultbyNTT(f,g,n,q,n_inv,NTTarray,INTTarray):
    return INTT(multiplyNTT(NTT(f,n,q,NTTarray),NTT(g,n,q,NTTarray),q),n,q,n_inv,INTTarray)

# Pt-NTT

In [17]:
# q is a prime and n is a power of two
# f and g are polynomials in Z_q[x]/(x^n-a)
# alpha is a non-negative integer such that full NTT is possible in (Z_q[y]/(y^(n/(2^alpha))-a), 
# i.e. a = w^(n/(2^alpha)*k) for some generator w of Z_q* and (n/(2^alpha))|(q-1)
# columns_inv is the multiplicative inverse of (n/(2^alpha)) modulo q
# NTTarray and INTTarray are the arrays created by the function compute_arrays(w,n/(2^alpha),q,k)

def polyMultbyPtNTT(f,g,n,q,a,alpha,columns_inv,NTTarray,INTTarray):
    # the rows are representing polynomials in Z_q[y]/(y^(n/(2^alpha))-c)
    rows = 2**alpha
    columns = n//rows
    f = np.reshape(f,shape=(rows,columns),order='F')
    g = np.reshape(g,shape=(rows,columns),order='F')
    
    # variable for computing the product
    prod = np.zeros(shape=(rows,columns),dtype=int)

    # computing y*f_l; it is only needed for l >= 1
    f_hat = np.roll(f,shift=1,axis=1)
    f_hat[1:,0] = (f_hat[1:,0]* a) % q
    #print(f_hat)

    # computing all the necessary NTTs and storing them in the same variables
    # the NTTs are computed in the ring Z_q[y]/(y^(n/(2^alpha))-c)
    f = np.array([NTT(matrix_row,columns,q,NTTarray) for matrix_row in f])
    g = np.array([NTT(matrix_row,columns,q,NTTarray) for matrix_row in g])

    if alpha > 0: # otherwise f_hat has only one row
        f_hat[1:] = np.array([NTT(matrix_row,columns,q,NTTarray) for matrix_row in f_hat[1:]])

    # computing the entries of prod in Z_q[y]/(y^(n/(2^alpha))-c)
    for i in range(rows):
        for l in range(i+1):
            prod[i] = addNTT(prod[i],multiplyNTT(f[l],g[i-l],q),q)
        for l in range(i+1,rows):
            prod[i] = addNTT(prod[i],multiplyNTT(f_hat[l],g[rows+i-l],q),q)

        prod[i] = INTT(prod[i],columns,q,columns_inv,INTTarray)

    # rewriting the result as a polynomial in x (vector instead of array)
    prod = np.reshape(prod,shape=-1,order='F')

    return prod

# K-NTT

In [18]:
# q is a prime and n is a power of two
# f and g are polynomials in Z_q[x]/(x^n-a)
# alpha is a non-negative integer such that full NTT is possible in Z_q[y]/(y^(n/(2^alpha))-a), 
# i.e. a = w^(n/(2^alpha)*k) for some generator w of Z_q* and (n/(2^alpha))|(q-1)
# columns_inv is the multiplicative inverse of (n/(2^alpha)) modulo q
# NTTarray and INTTarray are the arrays created by the function compute_arrays(w,n/(2^alpha),q,k)
# NTTy is the precomputed NTT of y, where y is in Z_q[y]/(y^(n/(2^alpha))-a) (using the same NTTarray)

def polyMultbykNTT(f,g,n,q,a,alpha,columns_inv,NTTarray,INTTarray,NTTy):
    # the rows are representing polynomials in Z_q[y]/(y^(n/(2^alpha))-a)
    rows = 2**alpha
    columns = n//rows
    f = np.reshape(f,shape=(rows,columns),order='F')
    g = np.reshape(g,shape=(rows,columns),order='F')
    
    # variable for computing the product
    prod = np.zeros(shape=(rows,columns),dtype=int)

    # computing all the necessary NTTs and storing them in the same variables
    # the NTTs are computed in the ring Z_q[y]/(y^(n/(2^alpha))-c)
    f = np.array([NTT(matrix_row,columns,q,NTTarray) for matrix_row in f])
    g = np.array([NTT(matrix_row,columns,q,NTTarray) for matrix_row in g])

    # precomputing the products of the NTT; pointwise product of the i-th row of f with
    # the i-th row of g for all rows i
    precomp = (f*g) % q

    # computing the entries of prod in Z_q[y]/(y^(n/(2^alpha))-a)
    # we first compute the entries of the product for rows with even indices
    for i in range(0,rows,2):
        # l is lower index, i-l is higher index, we want to compute (f[l]*g[i-l]+f[i-l]*g[l]) 
        # in one go using Karatsuba trick (the f[j] and g[j] are already in NTT form)
        for l in range(i//2):
            prod[i] = (addNTT(prod[i],multiplyNTT(addNTT(f[l],f[i-l],q),
                        addNTT(g[l],g[i-l],q),q)-precomp[l]-precomp[i-l],q))
        # adding the term f[i/2]*g[i/2]
        prod[i] = addNTT(prod[i],precomp[i//2],q)
        # temp_sum is the sum that is multiplied by NTTy before adding it to prod[i]
        # adding the term f[l]*g[l] where l = 2^alpha+i-l (which is (2^alpha+i)/2) to temp_sum
        # have to distinguish the case where alpha=0; then this sum is empty
        if alpha>0:
            temp_sum = precomp[(rows+i)//2]
        else:
            temp_sum = np.zeros(columns,dtype=int)
        for l in range(i+1,(rows+i)//2):
            temp_sum = (addNTT(temp_sum,multiplyNTT(addNTT(f[l],f[rows+i-l],q),
                        addNTT(g[l],g[rows+i-l],q),q)-precomp[l]-precomp[rows+i-l],q))
        # adding NTTy*temp_sum to prod[i] and applying the inverse NTT
        prod[i] = addNTT(prod[i],multiplyNTT(NTTy,temp_sum,q),q)
        prod[i] = INTT(prod[i],columns,q,columns_inv,INTTarray)

    # computing the entries of the product for rows with odd indices
    for i in range(1,rows,2):
        # l is lower index, i-l is higher index, we want to compute (f[l]*g[i-l]+f[i-l]*g[l]) 
        # in one go using Karatsuba trick (the f[j] and g[j] are already in NTT form)
        for l in range(i//2+1): # including l=i//2
            prod[i] = (addNTT(prod[i],multiplyNTT(addNTT(f[l],f[i-l],q),
                        addNTT(g[l],g[i-l],q),q)-precomp[l]-precomp[i-l],q))
        # temp_sum is the sum that is multiplied by NTTy before adding it to prod[i]
        temp_sum = np.zeros(columns,dtype=int)
        for l in range(i+1,(rows+i)//2+1):
            temp_sum = (addNTT(temp_sum,multiplyNTT(addNTT(f[l],f[rows+i-l],q),
                        addNTT(g[l],g[rows+i-l],q),q)-precomp[l]-precomp[rows+i-l],q))
        # adding NTTy*temp_sum to prod[i] and applying the inverse NTT
        prod[i] = addNTT(prod[i],multiplyNTT(NTTy,temp_sum,q),q)
        prod[i] = INTT(prod[i],columns,q,columns_inv,INTTarray)

    # rewriting the result as a polynomial in x (vector instead of array)
    prod = np.reshape(prod,shape=-1,order='F')

    return prod